In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [2]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="z-ai/glm-5.3-flash",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0
)

In [3]:
response = model.invoke(
    "Explain what an AI agent is in simple terms."
)

print(response.content)

ForbiddenResponseError: Key limit exceeded (total limit). Manage it using https://openrouter.ai/workspaces/default/keys/431afb6ccbdfc0b273aae9de4ead52e04d8354fbd165a01244f0ce9d93aea049

In [11]:
from langchain_core.tools import tool

In [12]:
@tool
def check_system_status(system: str) -> str:
    """
    Check the operational status of an enterprise system.
    """

    systems = {
        "wifi": "Wi-Fi service is operational.",
        "vpn": "VPN service is operational.",
        "email": "Email service is operational.",
        "github": "GitHub service is operational."
    }

    return systems.get(
        system.lower(),
        f"No status information is available for {system}."
    )

In [13]:
@tool
def get_employee_information(employee_name: str) -> str:
    """
    Retrieve employee information from the enterprise directory.
    """

    employees = {
        "rahul": {
            "department": "Finance",
            "office": "Hyderabad",
            "device": "Dell Latitude 5440"
        },
        "priya": {
            "department": "Engineering",
            "office": "Bangalore",
            "device": "Lenovo ThinkPad"
        },
        "arjun": {
            "department": "HR",
            "office": "Delhi",
            "device": "HP EliteBook"
        }
    }

    employee = employees.get(employee_name.lower())

    if not employee:
        return f"No employee information found for {employee_name}."

    return (
        f"Employee: {employee_name}\n"
        f"Department: {employee['department']}\n"
        f"Office: {employee['office']}\n"
        f"Device: {employee['device']}"
    )

In [14]:
@tool
def create_ticket(employee_name: str, issue: str) -> str:
    """
    Create an IT support ticket for an employee.
    """

    ticket_id = "INC-1001"

    return (
        f"Ticket created successfully.\n"
        f"Ticket ID: {ticket_id}\n"
        f"Employee: {employee_name}\n"
        f"Issue: {issue}\n"
        f"Status: Open"
    )

In [15]:
@tool
def search_it_policy(query: str) -> str:
    """
    Search the company's IT policies.
    """

    policies = {
        "password": (
            "Employees must change their password every 90 days. "
            "Passwords must contain uppercase, lowercase, numbers, "
            "and special characters."
        ),
        "vpn": (
            "Employees must use the approved company VPN when accessing "
            "internal systems from outside the corporate network."
        ),
        "software": (
            "Employees must request approval from IT before installing "
            "company-managed software."
        ),
        "wifi": (
            "Employees should connect company devices to the approved "
            "corporate Wi-Fi network."
        )
    }

    query = query.lower()

    for keyword, policy in policies.items():
        if keyword in query:
            return policy

    return "No matching IT policy was found."

In [17]:
@tool
def get_ticket_status(ticket_id: str) -> str:
    """
    Retrieve the current status of an IT support ticket.
    """

    tickets = {
        "INC-1001": "Open - IT technician has been assigned.",
        "INC-1002": "Resolved - Password was successfully reset.",
        "INC-1003": "In Progress - VPN issue is being investigated."
    }

    return tickets.get(
        ticket_id.upper(),
        f"Ticket {ticket_id} was not found."
    )

In [18]:
tools = [
    check_system_status,
    get_employee_information,
    create_ticket,
    get_ticket_status,
    search_it_policy
]

for tool in tools:
    print(tool.name)

check_system_status
get_employee_information
create_ticket
get_ticket_status
search_it_policy


In [19]:
model_with_tools = model.bind_tools(tools)

In [20]:
response = model_with_tools.invoke(
    "What is the status of ticket INC-1003?"
)

print(response.tool_calls)

[{'name': 'get_ticket_status', 'args': {'ticket_id': 'INC-1003'}, 'id': 'call_bcc82b35f3874a2689977c74', 'type': 'tool_call'}]


In [22]:
tool_call = response.tool_calls[0]

print("Tool Name:", tool_call["name"])
print("Arguments:", tool_call["args"])
print("Tool Call ID:", tool_call["id"])

Tool Name: get_ticket_status
Arguments: {'ticket_id': 'INC-1003'}
Tool Call ID: call_bcc82b35f3874a2689977c74


In [23]:
tool_result = get_ticket_status.invoke(
    tool_call["args"]
)

print(tool_result)

In Progress - VPN issue is being investigated.


In [24]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"]
)

print(tool_message)

content='In Progress - VPN issue is being investigated.' tool_call_id='call_bcc82b35f3874a2689977c74'


In [25]:
messages = [
    {
        "role": "user",
        "content": "What is the status of ticket INC-1003?"
    },
    response,
    tool_message
]

final_response = model_with_tools.invoke(messages)

print(final_response.content)

Ticket **INC-1003** is currently **In Progress** — the VPN issue is being actively investigated by the IT team.
